# ΒΗΜΑ 1 — Η τριάδα του gemma-2-2b: αστέρας και clique εδώ, δακτύλιος από το ablation

> **Σειρά εκτέλεσης:** αυτό είναι το **πρώτο** notebook. Μετά έρχεται το
> `kaggle_clique.ipynb` (βήματα 2–6). Το `kaggle_commfix_ablation.ipynb` έχει
> **ολοκληρωθεί** — μην το ξανατρέξεις.

**Γιατί το gemma-2-2b, και γιατί και οι τρεις τοπολογίες μαζί.** Είναι η
μοναδική αντιστροφή ολόκληρης της καμπάνιας: στο PD χωρίς επικοινωνία
συνεργάζεται **0,67 στον αστέρα και 0,18 στον δακτύλιο**. Αυτή η μία τιμή
κρατά το RQ2 στο 4+/1− (p ≈ 0,19) αντί για 5/5.

Όμως το 0,67 μετρήθηκε τον **Μάιο**, πριν το Kaggle περάσει σε transformers
5.x (28 Ιουλίου), ενώ ο δακτύλιος μετρήθηκε μετά. Η σύγκριση που κρατά την
αντιστροφή είναι λοιπόν πιθανά και σύγκριση δύο εκδόσεων βιβλιοθήκης.

Εδώ τρέχουν **αστέρας και clique**· ο δακτύλιος τρέχει τον ίδιο μήνα στο
`kaggle_commfix_ablation.ipynb`. Όλα τα records γράφουν την έκδοση
βιβλιοθήκης, οπότε ελέγχουμε ότι οι δύο συνεδρίες ταιριάζουν.

| ερώτημα | τι το κρίνει |
|---|---|
| Είναι πραγματική η αντιστροφή; | αστέρας έναντι δακτυλίου, χωρίς χρονικό κενό |
| Ισχύει η αραίωση εμπειρίας; | αστέρας < δακτύλιος < clique; |
| Κλείνει το ablation του prompt; | ο δακτύλιος του ablation είναι το commfix που έλειπε |

| έκβαση για το `no_comm` του αστέρα | συμπέρασμα |
|---|---|
| κοντά στο **0,67** | η αντιστροφή είναι πραγματική |
| κοντά στο **0,18** | ήταν τεχνούργημα βιβλιοθήκης — το RQ2 γίνεται 5/5 |

**Τι δεν αλλάζει.** PD, 16 γύροι, κρυφός ορίζοντας, μνήμη 10, T=0,7,
μηνύματα ως 20 λέξεις, `max_tokens` 256 όπως σε όλα τα gemma-2-2b runs.


## Setup

1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN`, **Attach to notebook**
   *(υποχρεωτικό — το gemma είναι gated. Αν λείπει, η συνεδρία αποτυγχάνει
   σε ~50 δευτερόλεπτα με `GatedRepoError 401`)*

Μετά: **Save Version → Save & Run All**.

**~4,6 ώρες, μία συνεδρία** (όριο Kaggle: 12 h ανά συνεδρία). Αν η συνεδρία
σου έχει μικρότερο όριο ή θες σιγουριά, τρέξε το σε **δύο** συνεδρίες:
`TOPOLOGIES = ['star']` στην πρώτη, `['clique']` στη δεύτερη. Αστέρας πρώτα, clique μετά, και η καθεμία ζιπάρεται μόλις τελειώσει. Αν κοπεί
η συνεδρία, ό,τι έχει τελειώσει είναι στο Output panel.


In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), 'GPU off -- Settings -> Accelerator -> GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
REPO_DIR = '/kaggle/working/repo'

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)

assert os.path.exists('campaign.py'), f'campaign.py δεν είναι στο {os.getcwd()}'
for f in ('campaign.py', 'run_all_scenarios.py'):
    assert '--topology-aware-comm-prompt' in open(f, encoding='utf-8').read(), (
        f'Το {f} δεν έχει --topology-aware-comm-prompt -- κάνε git push πρώτα.')
# Χωρίς αυτό, τα records δεν θα γράφουν έκδοση βιβλιοθήκης -- και ολόκληρο το
# νόημα αυτού του notebook είναι ότι τρέχει σε ΜΙΑ γνωστή έκδοση.
assert '_environment' in open('engine.py', encoding='utf-8').read(), \
    'Το engine.py δεν καταγράφει περιβάλλον -- κάνε git pull/push.'

print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())
import transformers
print('transformers:', transformers.__version__)


## Ο έλεγχος που κρίνει το πείραμα

Χτίζει το prompt και για τις τρεις τοπολογίες. Για τον **αστέρα** το
διορθωμένο prompt πρέπει να είναι **ταυτόσημο** με το παλιό — αλλιώς ο νέος
αστέρας δεν συγκρίνεται με τον Μάιο για λόγο άσχετο με τη βιβλιοθήκη. Για
**δακτύλιο** και **clique** πρέπει να αλλάζει ακριβώς μία γραμμή, και να μην
έχει μείνει κείμενο αστέρα.


In [ ]:
import sys
sys.path.insert(0, '.')
from games import GAMES
from topology import make_topology
from prompts import build_system_prompt

for name, want_neighbours in (('star', 3), ('cycle', 2), ('clique', 3)):
    topo = make_topology(name, 4)
    aid = 0
    n = len(topo.neighbors(aid))
    assert n == want_neighbours, (name, n)
    common = dict(game=GAMES['pd'], condition='cheap_talk', n_neighbors=n,
                  total_agents=4, topology_text=topo.describe(aid))
    old = build_system_prompt(**common)
    new = build_system_prompt(**common,
                              communication_text=topo.describe_communication(aid))
    diff = [l for l in new.splitlines() if l not in old.splitlines()]
    if name == 'star':
        assert old == new, 'ο αστέρας ΠΡΕΠΕΙ να έχει ίδιο prompt με τον Μάιο -- σταμάτα'
        print(f'{name:7s} prompt ταυτόσημο με το παλιό  (σωστό)')
    else:
        assert len(diff) == 1, f'{name}: άλλαξαν {len(diff)} γραμμές -- σταμάτα'
        assert 'central agent' not in new and 'peripheral' not in new, \
            f'{name}: έμεινε κείμενο αστέρα -- σταμάτα'
        print(f'{name:7s} αλλάζει μία γραμμή, χωρίς κείμενο αστέρα  (σωστό)')


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token φορτώθηκε')
except Exception as e:
    raise SystemExit(f'Χωρίς HF_TOKEN το gemma δεν φορτώνεται -- σταμάτα: {e}')


## Τι θα τρέξει

Ίδια τέσσερα σενάρια σε κάθε τοπολογία, **PD μόνο, n=5**. Ο δακτύλιος έρχεται από το ablation (βλ. κελί ρυθμίσεων): **~4,6 ώρες** αντί για 6,5.

| τοπολογία | runs | κλήσεις | ώρες | φάκελος |
|---|---|---|---|---|
| star | 25 | 2.240 | ~1,9 | `gemma-2-2b-it_commfix` |
| clique | 25 | 2.240 | ~2,7 | `gemma-2-2b-it_clique_commfix` |

Η σημαία `--topology-aware-comm-prompt` μπαίνει και στον αστέρα: δεν αλλάζει
το prompt του (το είδαμε μόλις), αλλά στέλνει τα runs σε **δικό τους** φάκελο,
ώστε να μη μπλεχτούν ποτέ με τα runs του Μαΐου.


In [ ]:
MODEL      = 'google/gemma-2-2b-it'
# Ο δακτύλιος ΔΕΝ είναι εδώ: τον μετρά το kaggle_commfix_ablation.ipynb,
# που έτρεξε το gemma-2-2b ξεχωριστά τον Σεπτέμβριο. Και οι δύο συνεδρίες
# γράφουν την έκδοση βιβλιοθήκης στα records, οπότε ελέγχουμε ότι ταιριάζουν.
# ΑΝ ΕΚΕΙΝΟ ΤΟ RUN ΑΠΕΤΥΧΕ, ξεσχολίασε το 'cycle' παρακάτω.
TOPOLOGIES = ['star', 'clique']                 # + 'cycle' αν χρειαστεί
SCENARIOS  = ['silence', 'no_sense', 'counterfactual', 'baseline']
GAMES      = ['pd']
EXPECTED   = 25                                 # 5+5+5+10 ανά τοπολογία
# Η clique ξεκινά μόνο αν ο αστέρας τελείωσε ως τότε: θέλει ~2,7-3,5 h, άρα
# με 4,5 h ορίου τελειώνει μέσα σε 8 h. Αλλιώς παραλείπεται καθαρά, και την
# τρέχεις σε νέα συνεδρία με TOPOLOGIES = ['clique'].
WALL_HOURS = 4.5

def args_for(topology):
    return ['--model', MODEL, '--session', 'A', '--topology', topology,
            '--scenarios', *SCENARIOS, '--games', *GAMES,
            '--topology-aware-comm-prompt']

for t in TOPOLOGIES:
    print(t, ' '.join(args_for(t)))


## Preview — κοστίζει μηδέν

Για κάθε τοπολογία πρέπει να δεις τη σωστή `topology`, τη γραμμή `PROMPT`,
`expecting : 25 run files` και φάκελο που τελειώνει σε `_commfix`.


In [ ]:
import subprocess, sys

ok = True
for t in TOPOLOGIES:
    print('=' * 70)
    r = subprocess.run([sys.executable, 'campaign.py', *args_for(t), '--dry-run'],
                       capture_output=True, text=True)
    out = r.stdout.strip()
    print(out[:900])
    if r.returncode != 0:
        ok = False
        print('ΣΦΑΛΜΑ:', r.stderr.strip()[-300:])
    for needle in (f'topology     : {t}', 'topology-aware', '_commfix',
                   f'expecting    : {EXPECTED}'):
        if needle not in out:
            ok = False
            print(f'ΛΕΙΠΕΙ ΑΠΟ ΤΟ ΠΛΑΝΟ: {needle!r}')

assert ok, 'Κάποιο plan απέτυχε -- μη συνεχίσεις.'
print('\n' + '=' * 70)
print('όλα τα πλάνα εντάξει')


## Εκτέλεση

Μία τοπολογία τη φορά. Αν μία αποτύχει, οι επόμενες συνεχίζουν.


In [ ]:
import subprocess, sys, time

t0 = time.time()
results = []
for t in TOPOLOGIES:
    elapsed_h = (time.time() - t0) / 3600
    if elapsed_h > WALL_HOURS:
        print(f'\n[STOP] {elapsed_h:.1f} h -- δεν ξεκινάω το {t}, δεν προλαβαίνει.')
        results.append((t, 'skipped', 0.0))
        continue
    print('\n' + '=' * 70)
    print(f'{MODEL}  {t}   ({elapsed_h:.1f} h μέχρι τώρα)')
    print('=' * 70, flush=True)
    s = time.time()
    r = subprocess.run([sys.executable, 'campaign.py', *args_for(t)])
    mins = (time.time() - s) / 60
    status = 'OK' if r.returncode == 0 else f'FAILED (exit {r.returncode})'
    results.append((t, status, mins))
    print(f'\n--> {t}: {status}, {mins:.0f} λεπτά', flush=True)

print('\n' + '=' * 70)
print('ΑΠΟΛΟΓΙΣΜΟΣ')
for t, s, mins in results:
    print(f'  {s:22s} {mins:6.0f} λ   {t}')
print(f'\nσύνολο {(time.time() - t0) / 3600:.1f} h')


## Το αποτέλεσμα, πριν καν κατεβάσεις τίποτα

Ελέγχει ότι κάθε record γράφει τη σωστή τοπολογία, το διορθωμένο prompt και
το περιβάλλον, και τυπώνει τη συνεργασία ανά τοπολογία και κελί — δηλαδή την
απάντηση και στα τρία ερωτήματα.


In [ ]:
import glob, json, collections, statistics, sys
sys.path.insert(0, '.')
from analysis import summarise_run, scenario_of

rows = collections.defaultdict(list)
envs = set()
for d in sorted(glob.glob('/kaggle/working/results/gemma-2-2b-it*_commfix')):
    for p in glob.glob(f'{d}/**/*.json', recursive=True):
        if os.path.basename(p).startswith('_'):
            continue
        rec = json.load(open(p, encoding='utf-8'))
        cfg = rec['config']
        assert cfg.get('topology_aware_comm_prompt') is True, f'ΛΑΘΟΣ PROMPT: {p}'
        assert rec.get('environment'), f'ΧΩΡΙΣ ΠΕΡΙΒΑΛΛΟΝ: {p}'
        env = rec['environment']
        envs.add((env.get('transformers'), env.get('torch'), env.get('gpu')))
        topo = rec['topology'].get('type', 'star')
        rows[(scenario_of(rec), topo)].append(summarise_run(rec)['coop_rate_overall'])

print('περιβάλλον:', envs, '\n')
cells = ['no_comm', 'silence', 'no_sense', 'counterfactual', 'baseline_cheap_talk']
print(f"{'cell':22s} {'star':>8s} {'cycle':>8s} {'clique':>8s}   μονότονα;")
for c in cells:
    v = {t: statistics.fmean(rows[(c, t)]) if rows.get((c, t)) else float('nan')
         for t in ('star', 'cycle', 'clique')}
    mono = v['star'] <= v['cycle'] <= v['clique']
    print(f"{c:22s} {v['star']:8.3f} {v['cycle']:8.3f} {v['clique']:8.3f}   {'ναι' if mono else 'όχι'}")

print('\nαναφορά, αστέρας του Μαΐου:  no_comm 0.675  (runs 0.11 0.52 0.89 0.89 0.97)')
print('αναφορά, δακτύλιος Ιουλίου:  no_comm 0.178')


## Μάζεμα


In [ ]:
import glob, os, shutil

zips = sorted(glob.glob('/kaggle/working/gemma-2-2b-it_*commfix*.zip'))
print(f'{len(zips)} zip:')
for z in zips:
    print(f'   {os.path.getsize(z)/1e6:6.1f} MB  {os.path.basename(z)}')
if zips:
    box = '/kaggle/working/g2b_triad'
    os.makedirs(box, exist_ok=True)
    for z in zips:
        shutil.copy(z, box)
    out = shutil.make_archive('/kaggle/working/g2b_triad_all', 'zip', box)
    print(f'\nκατέβασε αυτό: {out}  ({os.path.getsize(out)/1e6:.1f} MB)')
else:
    print('\nΚΑΝΕΝΑ ZIP -- δες τον απολογισμό, και ψάξε τους φακέλους '
          'results/gemma-2-2b-it*_commfix στο Output panel.')


## Μετά

Κατέβασε το `g2b_triad_all.zip` στο `diplomatikh/g2b_triad/`. **Ανάλυσέ το
μόνο του** — μη δώσεις ποτέ μαζί στο ίδιο `--roots` τον νέο αστέρα και τον
`gemma-2-2b-it_star` του Μαΐου: το `no_comm` δεν παίρνει ετικέτα γενιάς, οπότε
τα δύο θα ενώνονταν σε ένα κελί. Η σύγκριση με τον Μάιο γίνεται με το
`commfix_report.py`, που δέχεται παλιά και νέα δέντρα χωριστά.

Επόμενο: **`kaggle_clique.ipynb`, βήμα 2.**
